# Phase 08 — Spatial Simulator (Précalcul Offline)

> **📁 Résultat de cette phase :** Les matrices de fractions de transfert géométriques sont sauvegardées dans `research/offline/fractions.parquet` et `research/offline/fractions_1024.parquet`.

Ce notebook calcule les *fractions de transfert* pour chaque cellule. Il s'agit du cœur de l'architecture spatiale v2.0.

## 🔬 Principe physique : pourquoi remplacer le modèle précédent ?
Dans l'approche précédente, l'angle de handover était fixe et la fonction de transfert f(δ) était linéaire. C'était trop simpliste. La nouvelle approche est **spatiale et déterministe** :
- On discrétise chaque cellule en une grille 50×50 points.
- Pour chaque point, on calcule la qualité signal (path-loss = −3.76 × log₁₀(distance)) vers l'antenne maître et les voisines.
- Un point bascule vers une voisine si : `Q_voisine + δ >= Q_maitre`.
- La fraction transférée est le rapport des points qui basculent.

## ⚙️ Logique et Workflow
- **Indépendance** : Ce calcul est purement géométrique. Il ne dépend pas du trafic. 
- **Offline** : Effectué une fois (ou quand la topologie change), ce qui permet une résolution rapide online.
- **A3 Offset (δ)** : Nous précalculons ces fractions pour δ ∈ {0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0} dB pour couvrir tout le spectre d'action SON.

In [ ]:
import numpy as np
import polars as pl
import json
import yaml
from pathlib import Path

GRID_SIZE = 100
PL_EXP = 3.76
DELTA_LEVELS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

class SpatialTransferSimulator:
    def __init__(self, grid_resolution=50, pl_exp=3.76, grid_size=100):
        self.grid = grid_resolution
        self.pl_exp = pl_exp
        self.grid_size = grid_size

    def precompute_offline(self, coverage, antennas, neighbors, delta_levels):
        records = []
        total_masters = len(coverage)
        curr_master = 0
        
        for master_id, cell_list in coverage.items():
            curr_master += 1
            if curr_master % 20 == 0: print(f"Antenne {curr_master}/{total_masters}...")
                
            ax, ay = antennas[master_id]['x'], antennas[master_id]['y']
            voisines = neighbors.get(master_id, [])
            
            if not voisines:
                for square_id in cell_list:
                    for k, delta in enumerate(delta_levels):
                        records.append({'square_id': int(square_id), 'master_id': master_id, 'target_ant': master_id, 'delta_level': k, 'delta_dB': delta, 'fraction': 1.0})
                continue

            for square_id in cell_list:
                square_id = int(square_id)
                row, col = (square_id - 1) // self.grid_size, (square_id - 1) % self.grid_size
                xs, ys = np.linspace(col, col + 1, self.grid), np.linspace(row, row + 1, self.grid)
                X, Y = np.meshgrid(xs, ys)
                
                Q_master = -self.pl_exp * np.log10(np.hypot(X - ax, Y - ay) + 1e-6)
                best_Q, best_voisine = np.full_like(Q_master, -np.inf), np.full(Q_master.shape, '', dtype=object)
                
                for b_id in voisines:
                    Q_b = -self.pl_exp * np.log10(np.hypot(X - antennas[b_id]['x'], Y - antennas[b_id]['y']) + 1e-6)
                    mask = Q_b > best_Q
                    best_Q, best_voisine = np.where(mask, Q_b, best_Q), np.where(mask, b_id, best_voisine)
                
                delta_crit = Q_master - best_Q
                for k, delta in enumerate(delta_levels):
                    switch_mask = (delta >= delta_crit) & (best_voisine != '')
                    records.append({'square_id': square_id, 'master_id': master_id, 'target_ant': master_id, 'delta_level': k, 'delta_dB': delta, 'fraction': float(np.mean(~switch_mask))})
                    for b_id in voisines:
                        frac_b = float(np.mean(switch_mask & (best_voisine == b_id)))
                        if frac_b > 0: records.append({'square_id': square_id, 'master_id': master_id, 'target_ant': b_id, 'delta_level': k, 'delta_dB': delta, 'fraction': frac_b})
        return pl.DataFrame(records)

print("Chargement des données...")
with open('../config/network_topology.yaml', 'r') as f: antennas = yaml.safe_load(f)['antennas']
with open('../data/processed/cell_antenna_map_600.json', 'r') as f: coverage_600 = json.load(f)
with open('../data/processed/neighbor_graph.json', 'r') as f: neighbor_graph = json.load(f)

sim = SpatialTransferSimulator()
fractions_df = sim.precompute_offline(coverage_600, antennas, neighbor_graph, DELTA_LEVELS)

Path('../offline').mkdir(exist_ok=True)
fractions_df.write_parquet('../offline/fractions.parquet')
print(f"Précalcul terminé : {len(fractions_df)} lignes générées.")